# [M1] NÓI CHUYỆN VỚI DỮ LIỆU BẰNG PYTHON
## Buổi S04: Thống Kê Mô Tả, Phân Tích Nhóm (GroupBy) & Visualization

---

### Metadata Khóa Học
* **Module:** M1 — NÓI CHUYỆN VỚI DỮ LIỆU BẰNG PYTHON
* **Buổi:** S04
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook trực quan hóa đa chiều và phân tích thống kê nhóm
* **Mức độ liên quan VOAI / Olympic AI:** Phát hiện mô thức ẩn trong dữ liệu thi đấu, thiết kế feature thông minh.

#### Mục tiêu học tập (Learning Objectives)
- Tính toán và phân tích sâu các chỉ số thống kê mô tả: Mean, Median, Standard Deviation, Skewness, IQR.
- Sử dụng thành thạo cơ chế `Split-Apply-Combine` thông qua hàm `.groupby()` và `.agg()` trong Pandas.
- Trực quan hóa phân phối biến đơn lẻ (Univariate) bằng Histogram và Boxplot.
- Trực quan hóa mối quan hệ giữa các biến (Bivariate) bằng Scatter Plot và Heatmap ma trận tương quan.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M1_S02 và M1_S03.
- Hiểu các khái niệm thống kê cơ bản ở bậc phổ thông (trung bình, phương sai).

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Một bảng dữ liệu 100,000 dòng là một "mớ bòng bong" nếu bạn chỉ nhìn vào các con số thô. Làm thế nào để biết khách hàng nam hay nữ chi tiêu nhiều hơn? Làm thế nào để biết thời gian làm bài thi có tỷ lệ nghịch với số lỗi sai hay không?

Não bộ con người tiến hóa để xử lý hình ảnh và mô thức (patterns), không phải để đọc từng con số. **Thống kê mô tả và Trực quan hóa dữ liệu (Visualization)** chính là chiếc kính hiển vi giúp bạn nhìn thấu bản chất dữ liệu: tìm ra quy luật phân phối, phát hiện ra các nhóm hành vi tiềm năng, và định hình hướng đi đúng đắn cho các bước kỹ nghệ đặc trưng tiếp theo.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Cơ chế Split - Apply - Combine của GroupBy
* **Split:** Chia nhỏ DataFrame thành các nhóm con dựa trên một hoặc nhiều khóa phân loại (ví dụ: nhóm theo `City` hoặc `Gender`).
* **Apply:** Áp dụng một hàm tính toán độc lập lên từng nhóm con (ví dụ: tính trung bình điểm, đếm số lượng phần tử, tìm giá trị lớn nhất).
* **Combine:** Gộp kết quả tính toán từ tất cả các nhóm lại thành một cấu trúc dữ liệu tổng hợp duy nhất.

Cú pháp chuẩn mực trong thi đấu:
```python
df.groupby('Category_Col').agg({
    'Numeric_Col1': ['mean', 'median'],
    'Numeric_Col2': ['count', 'std']
})
```

### 2. Các loại biểu đồ trực quan hóa cốt lõi
1. **Histogram & KDE (Kernel Density Estimation):** Xem hình dạng phân phối của một biến số (có đối xứng chuẩn không, có bị lệch trái hay lệch phải không).
2. **Boxplot (Biểu đồ hộp Tukey):** Thể hiện trực quan 5 con số thống kê: $Min, Q1 (25\%), Median (50\%), Q3 (75\%), Max$ và phát hiện các chấm ngoại lai (Outliers).
3. **Scatter Plot (Biểu đồ phân tán):** Quan sát tương quan phi tuyến hoặc tuyến tính giữa hai biến số liên tục.
4. **Heatmap Correlation Matrix:** Thể hiện ma trận hệ số tương quan Pearson ($r \in [-1, 1]$) giữa tất cả các cặp đặc trưng số học.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Cấu trúc Boxplot (Tukey's Box):
          Q1 - 1.5*IQR                Q1         Median (Q2)       Q3             Q3 + 1.5*IQR
               │                      ┌──────────────┬─────────────┐                   │
  [Outliers] * ├─── Lower Whisker ────┤              │             ├──── Upper Whisker ┤ * [Outliers]
                                      └──────────────┴─────────────┘
                                      │<──────── IQR ─────────────>│
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Khoi tao tap du lieu mo phong ve nang suat hoc tap cua 200 sinh vien
np.random.seed(42)
n = 200

study_hours = np.random.uniform(2, 20, size=n)
# Diem thi phu thuoc tuyen tinh vao so gio hoc + nhieu ngau nhien
exam_scores = np.clip(30 + study_hours * 3.2 + np.random.normal(0, 5, size=n), 0, 100)
majors = np.random.choice(['AI', 'Data_Science', 'Computer_Science'], size=n, p=[0.4, 0.3, 0.3])
sleep_hours = np.random.normal(7, 1.2, size=n).round(1)

df_students = pd.DataFrame({
    'Major': majors,
    'Study_Hours': study_hours.round(1),
    'Sleep_Hours': sleep_hours,
    'Exam_Score': exam_scores.round(1)
})

# Thong ke gom nhom theo chuyen nganh (GroupBy)
grouped_stats = df_students.groupby('Major').agg(
    Avg_Score=('Exam_Score', 'mean'),
    Median_Score=('Exam_Score', 'median'),
    Std_Score=('Exam_Score', 'std'),
    Total_Students=('Exam_Score', 'count')
).reset_index()

print("===== KET QUA PHAN TICH NHOM (GROUPBY AGGREGATION) =====")
print(grouped_stats)

### 🔍 Code Walkthrough (Khối 1)
Khối code minh họa kỹ thuật gom nhóm dữ liệu chuyên nghiệp:
- Tạo dataset 200 sinh viên với mối quan hệ thực nghiệm giữa số giờ học và điểm thi.
- Sử dụng cú pháp Named Aggregation trong `.agg()` để gán nhãn rõ ràng cho các cột thống kê đầu ra (`Avg_Score`, `Median_Score`, `Total_Students`).


In [ ]:
# 2. Truc quan hoa da chieu bang Matplotlib va Seaborn
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bieu do 1: Histogram + KDE phan phoi diem thi
sns.histplot(df_students['Exam_Score'], kde=True, ax=axes[0], color='#2b5c8f', bins=15)
axes[0].set_title('Phan phoi Diem thi (Histogram & KDE)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Exam Score')

# Bieu do 2: Boxplot so sanh diem thi giua cac chuyen nganh
sns.boxplot(x='Major', y='Exam_Score', data=df_students, ax=axes[1], palette='Set2')
axes[1].set_title('So sanh Diem thi giua cac Nganh (Boxplot)', fontsize=12, fontweight='bold')

# Bieu do 3: Scatter Plot the hien tuong quan giua Gio hoc va Diem thi
sns.scatterplot(x='Study_Hours', y='Exam_Score', hue='Major', data=df_students, ax=axes[2], alpha=0.8)
axes[2].set_title('Tuong quan: Gio hoc vs Diem thi', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Study Hours / Week')

plt.tight_layout()
plt.show()

### 🔍 Code Walkthrough (Khối 2)
Khối code vẽ 3 biểu đồ trực quan hóa dữ liệu chuẩn mực:
- **Histogram & KDE:** Đánh giá độ lệch và mật độ phân phối điểm số.
- **Boxplot:** So sánh độ biến động và phát hiện outlier giữa 3 chuyên ngành.
- **Scatter Plot:** Quan sát trực quan tương quan dương mạnh mẽ giữa thời gian học và kết quả thi.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Phân tích phân vị (Quantiles)
Sử dụng hàm `np.quantile` hoặc `pd.Series.quantile` để tìm giá trị phân vị $25\%, 50\%, 75\%$ của cột `Study_Hours`.

### Bài 2 (Easy): Tính ma trận tương quan Pearson
Tính ma trận tương quan giữa các cột số trong `df_students` bằng lệnh `df_students[['Study_Hours', 'Sleep_Hours', 'Exam_Score']].corr()`. Cho biết cặp biến nào có tương quan tuyến tính mạnh nhất.

### Bài 3 (Medium): Vẽ Heatmap tương quan
Sử dụng hàm `sns.heatmap(..., annot=True, cmap='coolwarm')` để trực quan hóa ma trận tương quan vừa tính ở Bài 2.

### Bài 4 (Medium): Gom nhóm nhiều tầng (Multi-level GroupBy)
Tạo thêm cột phân loại nhị phân `Well_Rested` (`True` nếu `Sleep_Hours >= 7.0`, ngược lại `False`). Thực hiện gom nhóm theo cả hai tiêu chí `['Major', 'Well_Rested']` và tính điểm thi trung bình của từng nhóm con.

### Bài 5 (Challenge): Phát hiện nghịch lý Simpson (Simpson's Paradox)
Tìm hiểu khái niệm Nghịch lý Simpson trong thống kê. Tạo một ví dụ nhỏ bằng Pandas chứng minh một xu hướng xuất hiện trong từng nhóm nhỏ nhưng lại biến mất hoặc đảo ngược khi gộp chung tất cả các nhóm lại.



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong thi đấu Olympic AI, trực quan hóa không phải để "làm cảnh" mà là công cụ **thám tử dữ liệu**. Khi bạn vẽ phân phối của nhãn (Target variable) và phát hiện nhãn bị lệch cực đoan (Right-skewed, ví dụ bài toán dự đoán doanh số), động tác tiếp theo của một thí sinh đẳng cấp là ngay lập tức áp dụng biến đổi hàm Logarit $\log(1 + x)$ để đưa phân phối về dạng hình chuông đối xứng, giúp các mô hình Linear Regression và Neural Network hội tụ tốt hơn gấp nhiều lần!



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Một thí sinh gom nhóm dữ liệu bằng `.groupby()` và cố gắng lưu file CSV nhưng bị mất cột nhóm hoặc nhận file có cấu trúc index kỳ lạ:

```python
# CODE DỄ GÂY LỖI CẤU TRÚC:
res = df_students.groupby('Major')['Exam_Score'].mean()
# res bay gio la mot Series co index la 'Major', khong con la DataFrame dang bang thong thuong!
# Neu dem merge voi bang khac se de sinh loi mismatch index.
```

**Nhiệm vụ của bạn:**
1. Giải thích tại sao việc thiếu `.reset_index()` sau groupby có thể gây khó khăn khi thao tác ghép bảng dữ liệu (merge/join).
2. Sửa lại dòng lệnh trên để luôn nhận được một DataFrame chuẩn với các cột tên rõ ràng.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Đại lượng nào sau đây đo lường mức độ bất đối xứng của một phân phối xác suất quanh giá trị trung bình của nó?
   - A. Standard Deviation (Độ lệch chuẩn)
   - B. Skewness (Hệ số bất đối xứng / Độ lệch)
   - C. Kurtosis (Độ nhọn)
   - D. Variance (Phương sai)

2. Khoảng tứ phân vị (Interquartile Range - IQR) được tính bằng công thức nào?
   - A. $IQR = Q3 - Q1$
   - B. $IQR = Max - Min$
   - C. $IQR = Mean - Median$
   - D. $IQR = Q2 - Q1$

3. Hệ số tương quan Pearson $r = -0.85$ giữa hai biến $X$ và $Y$ mang ý nghĩa gì?
   - A. $X$ và $Y$ không có bất kỳ mối quan hệ nào.
   - B. $X$ và $Y$ có mối quan hệ tương quan tuyến tính nghịch biến rất mạnh (khi $X$ tăng thì $Y$ giảm mạnh).
   - C. $X$ là nguyên nhân trực tiếp gây ra $Y$.
   - D. Dữ liệu bị lỗi tính toán.

4. Biểu đồ Boxplot sử dụng đường kẻ bên trong hộp (line inside box) để biểu diễn giá trị nào?
   - A. Giá trị trung bình (Mean)
   - B. Giá trị trung vị (Median / Q2)
   - C. Giá trị lớn nhất (Max)
   - D. Độ lệch chuẩn

5. Cơ chế hoạt động của hàm `.groupby()` trong Pandas tuân theo mô hình nào?
   - A. Map - Filter - Reduce
   - B. Split - Apply - Combine
   - C. Fit - Predict - Score
   - D. Load - Transform - Dump



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Nắm vững các chỉ số thống kê mô tả nâng cao và ý nghĩa của chúng.
  - Sử dụng thành thạo mô hình `Split-Apply-Combine` với `.groupby()` và `.agg()`.
  - Thành thạo vẽ các biểu đồ phân phối và tương quan: Histogram, Boxplot, Scatter, Heatmap.
* **What I can now do:**
  - Tự tin phân tích mối tương quan giữa các nhóm đặc trưng trong bảng dữ liệu lớn.
  - Trực quan hóa dữ liệu sạch đẹp, phục vụ đắc lực cho việc ra quyết định kỹ thuật.
* **What I still need to practice:**
  - Kết hợp toàn bộ kỹ năng của M1 để thực hiện **Phân tích dữ liệu đầu tiên (First Data Analysis Project)** ở Buổi 5.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Buổi tiếp theo (**M1_S05**) là buổi tổng kết đỉnh cao của Module 1: **Data Cleaning Tổng Hợp, Phát Hiện Dị Biệt (Anomalies) & First Data Analysis Report**! Bạn sẽ tự tay thực hiện một bài toán phân tích dữ liệu thực tế từ A đến Z, xuất bản báo cáo phân tích đầu tiên và nhận Chứng chỉ **Certificate of Python and Data Basics**!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M1**:
- Thống kê mô tả & số đo phân tán ✓
- Phân tích nhóm bằng GroupBy & Aggregation ✓
- Trực quan hóa dữ liệu (Visualization) bằng Matplotlib & Seaborn ✓

* **Session Deliverable:** Mã nguồn phân tích thống kê nhóm và bộ biểu đồ trực quan hóa dữ liệu
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
